# 02 — Preprocessing & Testfall-Entwicklung

Von `df_clean` zum modellierbaren Datensatz — mit dokumentierten Hypothesen.

**Story:**
1. Transformer anwenden → `ddc_primary_3digit`, `has_sdnb`
2. Hypothese 1: DDC 549 als Mineralogie-Label → **unzureichend**
3. Hypothese 2: SDNB 38 laut Konkordanz → **empirisch falsch**
4. Hypothese 3: SDNB 31 empirisch bestätigt → **Trainingsbasis**
5. Plots: Klassifikationsabdeckung, DDC-Verteilung, Sunburst
6. Retro-Kandidaten identifizieren und speichern

**Input:**  `data/processed/df_clean.parquet`

**Output:** `data/processed/df_transformed.parquet`

In [ ]:
import sys
from pathlib import Path

from core.classification_transform import ClassificationTransformer
from core.data_explorer import Marc21Explorer, DDC_MAIN
# from core.filter_theses import Filter

import pandas as pd
# import numpy as np
import plotly.express as px
# import plotly.graph_objects as go

PROJECT_ROOT   = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

## 1 — df_clean laden

In [ ]:
df_clean = pd.read_parquet(DATA_PROCESSED / "df_clean.parquet")

# Listenfelder nach Parquet-Roundtrip reparieren
LIST_COLS = ["082_a", "082_2", "083_a", "083_2", "sdnb_codes", "subjects"]
for col in LIST_COLS:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(
            lambda x: list(x)
            if hasattr(x, "__iter__") and not isinstance(x, str)
            else (x or [])
        )

print(f"Records: {len(df_clean):,}")
print(f"Spalten: {list(df_clean.columns)}")

In [ ]:
# Filter: Titel enthält 'mineral'
df_mineral = df_clean[df_clean['title'].str.contains('mineral', case=False, na=False)].copy()

# DDC-Notationen (082_a)
ddc_codes = (
    df_mineral['082_a']
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)
ddc_codes_sorted = sorted(ddc_codes)

# SNDB-Notationen (083_a und 084_a)
sndb_codes = pd.concat([
    df_mineral['083_a'],
    df_mineral['sdnb_codes']
]).dropna().astype(str).str.strip().unique()

sndb_codes_sorted = sorted(sndb_codes)

# Ausgabe
print("DDC (082_a) – sortiert:")
for code in ddc_codes_sorted:
    print(code)

print("\nSNDB (083_a / sdnb_codes) – sortiert:")
for code in sndb_codes_sorted:
    print(code)

## 2 — Transformer anwenden

In [ ]:
df_transformed = ClassificationTransformer.transform_dataset(df_clean)

print("Neue Spalten: ddc_primary_3digit, has_sdnb, is_geowiss")
print(f"\nHat DDC:  {df_transformed['ddc_primary_3digit'].ne('').sum():>8,}")
print(f"Hat SDNB: {df_transformed['has_sdnb'].sum():>8,} ({df_transformed['has_sdnb'].mean()*100:.1f}%)")

### 2.1 Titel-Filter für df_transformed

In [ ]:
# Filter: Mineral / Boden / Sediment in Titel oder Subjects
df_hits = df_transformed[
    df_transformed['title'].str.contains('mineral|erz', case=False, na=False) |
    df_transformed['subjects'].str.contains('mineral|erz', case=False, na=False)
].copy()

# Relevante Spalten auswählen
df_hits = df_hits[
    [
        'record_id',
        'author_name',
        'title',
        'publication_year',
        'subjects',
        'sdnb_codes',
        'ddc_primary_3digit'
    ]
]

display(df_hits.head(5))

In [ ]:
def has_value(x):
    if isinstance(x, list):
        return len(x) > 0
    return pd.notna(x) and x != ""

# Treffer-DataFrame
df_hits = df_transformed[
    df_transformed['title'].str.contains('mineral|boden|sediment', case=False, na=False) |
    df_transformed['subjects'].str.contains('mineral|boden|sediment', case=False, na=False)
].copy()

# Zählen
count_sdnb = df_hits['sdnb_codes'].apply(has_value).sum()
count_ddc = df_hits['ddc_primary_3digit'].apply(has_value).sum()
count_both = df_hits[
    df_hits['sdnb_codes'].apply(has_value) &
    df_hits['ddc_primary_3digit'].apply(has_value)
].shape[0]

print("Treffer gesamt:", len(df_hits))
print("mit SDNB (nicht leer):", count_sdnb)
print("mit DDC 3-Steller (nicht leer):", count_ddc)
print("mit beidem:", count_both)

In [ ]:
def first(x):
    if isinstance(x, list):
        return x[0] if len(x) > 0 else None
    return x

# Jahr flach machen
df_hits['publication_year'] = df_hits['publication_year'].apply(first)

# Verteilung
year_dist = df_hits['publication_year'].value_counts().sort_index()

print(year_dist)

#### 2.2 Plot: Titelbegriffe über Publikationsjahre verteilt

In [ ]:
# Plot
import matplotlib.pyplot as plt

# Plot größer
plt.figure(figsize=(14,6))
year_dist.plot(kind='bar')

plt.title('Treffer nach publication_year (Mineral/Erz)')
plt.xlabel('Jahr')
plt.ylabel('Anzahl')

# Nur jede n-te Beschriftung zeigen (z.B. jedes 5. Jahr)
xticks = range(0, len(year_dist.index), 5)
plt.xticks(xticks, year_dist.index[::5], rotation=45)

plt.show()

## 3 — Plot: Klassifikationsabdeckung über die Zeit

Kernbefund: zwei Systemwechsel — SDNB ~1970, DDC ~2003.

In [ ]:
explorer = Marc21Explorer(df_transformed)
fig = explorer.plot_coverage_by_year(year_min=1924, year_max=2024)
fig.update_layout(title="Klassifikationsabdeckung DNB-Hochschulschriften 1924–2024")
fig.show()

In [ ]:
# Filter: nur Datensätze mit *Mineral* im Titel
df_erz = df_transformed[
    df_transformed['title'].str.contains('Mineral', case=False, na=False)
].copy()

# Explorer auf gefilterten Daten
explorer_erz = Marc21Explorer(df_erz)

# Coverage-Plot
fig = explorer_erz.plot_coverage_by_year(year_min=1924, year_max=2024)
fig.update_layout(title="Klassifikationsabdeckung – Datensätze mit 'Mineral*' im Titel (1924–2024)")
fig.show()

In [ ]:
# Filter: nur Datensätze mit *Erz* im Titel
df_erz = df_transformed[
    df_transformed['title'].str.contains('erz', case=False, na=False)
].copy()

# Explorer auf gefilterten Daten
explorer_erz = Marc21Explorer(df_erz)

# Coverage-Plot
fig = explorer_erz.plot_coverage_by_year(year_min=1924, year_max=2024)
fig.update_layout(title="Klassifikationsabdeckung – Datensätze mit 'Erz' im Titel (1924–2024)")
fig.show()

In [ ]:
def has_value(x):
    if isinstance(x, list):
        return len(x) > 0
    return pd.notna(x) and x != ""

# Filter: Titel enthält Erz
df_erz = df_transformed[
    df_transformed['title'].str.contains('erz', case=False, na=False)
].copy()

total = len(df_erz)

# Kategorien
with_ddc = df_erz['ddc_primary_3digit'].apply(has_value).sum()
with_sdnb = df_erz['sdnb_codes'].apply(has_value).sum()
with_both = df_erz[
    df_erz['ddc_primary_3digit'].apply(has_value) &
    df_erz['sdnb_codes'].apply(has_value)
].shape[0]

without = total - len(df_erz[
    (df_erz['ddc_primary_3digit'].apply(has_value)) |
    (df_erz['sdnb_codes'].apply(has_value))
])

print("Gesamt mit 'Erz' im Titel:", total)
print("mit DDC:", with_ddc)
print("mit SNDB:", with_sdnb)
print("mit beidem:", with_both)
print("ohne Klassifikation:", without)

In [ ]:
display(df_erz[df_erz["is_geowiss"]])

## 4 — Plot: DDC-Hauptklassenverteilung

In [ ]:
ddc_dist = (
    df_transformed[df_transformed["ddc_primary_3digit"].str.len() > 0]
    .assign(ddc_main=lambda d: d["ddc_primary_3digit"].str[0] + "00")
    .groupby("ddc_main")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=True)
)
ddc_dist["label"] = ddc_dist["ddc_main"].apply(
    lambda x: f"{x} — {DDC_MAIN.get(x[0], '?')}"
)

fig = px.bar(
    ddc_dist, x="count", y="label", orientation="h",
    title="DDC-Hauptklassenverteilung (Gesamtkorpus)",
    labels={"count": "Anzahl Records", "label": ""},
    color="count", color_continuous_scale="Blues",
)
fig.update_layout(plot_bgcolor="white", height=480,
                  coloraxis_showscale=False)
fig.update_xaxes(showgrid=True, gridcolor="#EEEEEE")
fig.show()

## 5 — Plot: Sunburst Naturwissenschaften (500er)

Mineralogie (DDC 549) wird ab 2003 explizit klassifiziert — rot hervorgehoben.

In [ ]:
df_hier = explorer.build_ddc_hierarchy()
df_nat  = df_hier[df_hier["DDC_1"].str.startswith("Naturwissenschaften")].copy()
df_nat["highlight"] = df_nat["DDC_3"].str.startswith("549")

fig = px.sunburst(
    df_nat, path=["DDC_1", "DDC_2", "DDC_3"], values="count",
    title="Naturwissenschaften (DDC 500er) — Mineralogie 549 in rot",
    color="highlight",
    color_discrete_map={True: "#E53935", False: "#90CAF9"},
)
fig.update_traces(
    hovertemplate="<b>%{label}</b><br>Records: %{value:,}<extra></extra>"
)
fig.update_layout(height=560)
fig.show()

n_549 = df_nat[df_nat["highlight"]]["count"].sum()
print(f"DDC-549-Records im Sunburst: {n_549:,}")
print("→ Alle ab 2003 — kein SDNB-Pendant vorhanden")

## 6 — Hypothese 1: DDC 549 als Mineralogie-Label

**Erwartung:** DDC 549 = Mineralogie, viele Records, gute Trainingsbasis.

**Befund:** Nur 216 Records, alle ab 2003 — zu wenig und zeitlich weit vom Retro-Ziel (1920–1970) entfernt.

In [ ]:
df_549 = df_transformed[
    df_transformed["ddc_primary_3digit"].str.startswith("549", na=False)
].copy()

print(f"DDC-549-Records: {len(df_549):,}")
print("\nNach Jahrzehnt:")
print(
    (df_549["publication_year"] // 10 * 10)
    .astype("Int16").value_counts().sort_index().to_string()
)

# SDNB-Überlapp prüfen
sdnb_bei_549 = df_549["sdnb_codes"].explode().dropna()
print(f"\nSDBN-Codes bei DDC-549-Records: {len(sdnb_bei_549):,}")
print("→ Kein SDNB-Pendant — Systeme überlappen nicht, sie lösen sich ab")
print("\n✗ Hypothese 1 verworfen: zu wenige Records, Domain-Shift zu groß")

## 7 — Hypothese 2: SDNB 38 laut DNB-Konkordanz

**Erwartung:** DNB-Sachgruppensystematik weist SDNB 38 als Geowissenschaften/Mineralogie aus.

**Befund:** Empirische Prüfung zeigt Wirtschaftswissenschaften.

In [ ]:
df_38 = df_transformed[
    df_transformed["sdnb_codes"].apply(
        lambda x: any(c.startswith("38") for c in x)
        if isinstance(x, list) else False
    )
].copy()

print(f"SDNB-38-Records: {len(df_38):,}")
print("\nTop DDC-Codes:")
print(df_38["ddc_primary_3digit"].value_counts().head(8).to_string())
print("\nBeispiel-Titel:")
print(df_38["title"].sample(8, random_state=42).to_string())
print("\n✗ Hypothese 2 verworfen: SDNB 38 = Wirtschaftswissenschaften (DDC 330/658)")

## 8 — Hypothese 3: SDNB 31 empirisch bestätigt

**Ansatz:** Welcher SDNB-Code dominiert bei Titeln mit Geo-Terminologie?

**Befund:** SDNB 31 mappt empirisch auf DDC 550 — Geowissenschaften bestätigt.

In [ ]:
# Empirisches SDNB→DDC-Mapping bei Records mit beiden Systemen
df_both = df_transformed[
    df_transformed["has_sdnb"] &
    (df_transformed["ddc_primary_3digit"].str.len() > 0)
].copy()
df_both["sdnb_2"] = df_both["sdnb_codes"].apply(
    lambda x: x[0][:2] if isinstance(x, list) and x else ""
)

geo_mapping = (
    df_both.groupby(["sdnb_2", "ddc_primary_3digit"])
    .size()
    .reset_index(name="count")
    .sort_values(["sdnb_2", "count"], ascending=[True, False])
    .groupby("sdnb_2").first().reset_index()
)
print("SDNB-Codes die auf DDC 55x mappen:")
print(
    geo_mapping[
        geo_mapping["ddc_primary_3digit"].str.startswith("55", na=False)
    ][["sdnb_2", "ddc_primary_3digit", "count"]].to_string()
)

In [ ]:
# SDNB-31-Records prüfen
df_31 = df_transformed[
    df_transformed["sdnb_codes"].apply(
        lambda x: any(c[:2] == "31" for c in x)
        if isinstance(x, list) else False
    )
].copy()

print(f"SDNB-31-Records: {len(df_31):,}")
print("\nNach Jahrzehnt:")
print(
    (df_31["publication_year"] // 10 * 10)
    .astype("Int16").value_counts().sort_index().to_string()
)
print("\nBeispiel-Titel:")
print(df_31["title"].sample(8, random_state=42).to_string())
print("\n✓ Hypothese 3 bestätigt: SDNB 31 = Geowissenschaften/Umweltgeologie")
print("  15k Records, 1970–2003, zeitlich nah am Retro-Ziel 1920–1970")

In [ ]:
# Zeigt die Generalisierbarkeit des Ansatzes
is_geowiss    = ClassificationTransformer.make_topic_flag(["549", "55"], ["31"])
is_geschichte = ClassificationTransformer.make_topic_flag(["9"], ["64"])
is_chemie     = ClassificationTransformer.make_topic_flag(["54"], ["30"])

print(f"Geowiss.:    {df_transformed.apply(is_geowiss, axis=1).sum():,}")
print(f"Geschichte:  {df_transformed.apply(is_geschichte, axis=1).sum():,}")
print(f"Chemie:      {df_transformed.apply(is_chemie, axis=1).sum():,}")

## 9 — Retro-Kandidaten identifizieren

Unklassifizierte Records 1920–1970 mit geowissenschaftlichem Textindiz.

In [ ]:
TERMS_GEO = {
    "geologie", "geochemie", "mineralogie", "petrologie",
    "lagerstätte", "sediment", "gestein", "kristallographie",
    "petrographie", "hydrogeologie", "grundwasser",
    "mineralisation", "mineralisierung", "erzlagerstätte",
}

no_class = (
    (df_transformed["ddc_primary_3digit"].fillna("") == "") &
    (~df_transformed["has_sdnb"])
)

def has_geo_term(row):
    text = " ".join([
        str(row.get("title", "")),
        str(row.get("title_remainder", "")),
        " ".join(row.get("subjects", [])),
    ]).lower()
    return any(t in text for t in TERMS_GEO)

df_retro = df_transformed[no_class].copy()
df_retro = df_retro[df_retro.apply(has_geo_term, axis=1)]
df_retro = df_retro.drop_duplicates(subset=["title"])
df_retro["decade"] = (
    df_retro["publication_year"] // 10 * 10
).astype("Int16")

print(f"Unklassifizierte Records gesamt:        {no_class.sum():,}")
print(f"Davon mit Geo-Textindiz (deduplic.):    {len(df_retro):,}")
print("\nNach Jahrzehnt:")
print(df_retro["decade"].value_counts().sort_index().to_string())

In [ ]:
# Plot: Retro-Bedarf
fig = explorer.plot_retro_bedarf()
fig.show()

## 10 — df_transformed speichern

In [ ]:
out_path = DATA_PROCESSED / "df_transformed.parquet"
df_transformed.to_parquet(out_path, engine="pyarrow", compression="snappy")
print(f"Gespeichert: {out_path}")
print(f"Größe:       {out_path.stat().st_size / 1024**2:.1f} MB")
print("\n=== Zusammenfassung ===")
print(f"Records gesamt:          {len(df_transformed):,}")
print(f"Mit DDC:                 {df_transformed['ddc_primary_3digit'].ne('').sum():,}")
print(f"Mit SDNB:                {df_transformed['has_sdnb'].sum():,}")
print(f"Unklassifiziert:         {no_class.sum():,}")
print(f"Retro-Kandidaten (Geo):  {len(df_retro):,}")
print(f"Trainingsbasis (SDNB 31): {len(df_31):,}")